Imports and config (always run this first)

In [12]:
import sys
sys.path.append('/mnt/md0/tempFolder/samAnderson/unet-gnn/') 
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from config import precursor_config as cfg
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Get the age and sex distributions of the datasets

In [ ]:
# Imports
import os
import pandas as pd
from glob import glob

# Function for getting dataset statistics based on what files are available
def get_dataset_statistics(cfg, age_filter=None):
    
    # Initialize the summary DataFrame with correct columns
    d_table = pd.DataFrame(columns=['repository', 'set', 'N_subj', 'N_scans', 'min', 'max', 'μ', 'σ', 'M:F'])

    # Store the training demographic data for depicting the combined set
    training_data = []

    # Define the raw file types
    file_types = [cfg.feature_map_dict[f] for f in cfg.features]
    hemispheres = ['_lh', '_rh']
    file_suffixes = [
        f"{hemi}.{feature}" if feature == "w-g.pct.mgh" else f"{hemi}_{feature}.mgh"
        for feature in file_types
        for hemi in hemispheres
    ]

    # Begin counting number of males and females
    training_sex_counts = {'Female': 0, 'Male': 0}
    
    for name, dset in cfg.datasets.items():
        # Get the metadata
        if dset['metadata'].endswith('.csv'):
            df = pd.read_csv(dset['metadata'])
        elif dset['metadata'].endswith('.xlsx'):
            df = pd.read_excel(dset['metadata'])
        else:
            raise Exception('Error: metadata must be csv or xlsx')
        
        # Apply preprocessing if specified
        if dset['data_preproc'] is not None:

            if 'DD/MM/YY conversion' in dset['data_preproc']:
                df[dset['date_col']] = df[dset['date_col']].apply(
                    lambda x: (
                        f"20{x.split('/')[2].zfill(2)}"
                        f"{x.split('/')[0].zfill(2)}"
                        f"{x.split('/')[1].zfill(2)}"
                    ) if pd.notna(x) else None
                )
        
            if 'remove_-' in dset['data_preproc']:
                df[dset['date_col']] = df[dset['date_col']].astype(str).str.replace('-', '', regex=False)
                
            if 'remove_.0' in dset['data_preproc']:
                df[dset['date_col']] = df[dset['date_col']].astype(str).str.replace(r"\.0$", "", regex=True)

            if 'full_date' in dset['data_preproc']:
                df[dset['date_col']] = pd.to_datetime(df[dset['date_col']], format='%m/%d/%Y').dt.strftime('%Y%m%d').astype(int)

            if 'all_str' in dset['data_preproc']:
                df[dset['date_col']] = df[dset['date_col']].astype(str)
                df[dset['sex_col']] = df[dset['sex_col']].astype(str)
                df[dset['id_col']] = df[dset['id_col']].astype(str)

        # Find raw files and determine split position
        raw_files = glob(f'{dset["raw_data"]}*')
        parts = raw_files[0][len(dset["raw_data"]):].split('_')
        split_position = None

        max_attempts = 5
        for attempt in range(1, max_attempts + 1):
            for i in range(attempt, len(parts)):
                potential_id = '_'.join(parts[:i])
                potential_date = parts[i]
                
                id_match = potential_id in df[dset['id_col']].values.astype(str)
                if dset['date_col'] is not None:
                    date_match = potential_date in df[dset['date_col']].values.astype(str)

                if potential_date == '00000000':
                    date_match = True
                
                if id_match and date_match:
                    split_position = i + 1
                        
        if split_position is None:
            raise Exception('Error: Aberrant relationship between raw file names and metadata IDs/dates')
        
        # Extract subject-date combinations and collect associated files
        subject_date_files = {}
        
        for f in raw_files:
            basename = os.path.basename(f)
            parts = basename.split('_')
            subj_date = '_'.join(parts[:split_position])
            
            if subj_date not in subject_date_files:
                subject_date_files[subj_date] = []
            
            for suffix in file_suffixes:
                if suffix in basename:
                    subject_date_files[subj_date].append(suffix)
                    break

        # Only keep subjects that have ALL required files   
        to_remove = []
        for key in subject_date_files.keys():
            for suffix in file_suffixes:
                if suffix not in subject_date_files[key]:
                    to_remove.append(key)
                    break
        for r in to_remove: 
            subject_date_files.pop(r) 

        subj_timepoints = [k for k in subject_date_files.keys()]
        
        # Keep only selected subject groups
        if dset['select'] != 'all':
            column, valid_val = dset['select'].split("==", 1)
            valid_val = str(valid_val).strip()
            df = df[df[column].astype(str).str.strip() == valid_val]   

        # Apply age limiting if specified
        if age_filter is not None:
            if age_filter < 0:
                df = df[df[dset['age_col']] < abs(age_filter)]
            else:
                df = df[df[dset['age_col']] >= age_filter]
            if df.empty: 
                continue

        # Filter DataFrame to include only valid subjects
        if dset['date_col'] is not None: 
            mask = df.apply(lambda row: f"{row[dset['id_col']]}_{row[dset['date_col']]}" in subj_timepoints, axis=1)
        else:
            mask = df.apply(lambda row: f"{row[dset['id_col']]}_00000000" in subj_timepoints, axis=1)

        filtered_df = df[mask]

        # Remove duplicates
        if dset['date_col'] is not None: 
            filtered_df = filtered_df.drop_duplicates(subset=[dset['id_col'], dset['date_col']])
        else:
            filtered_df = filtered_df.drop_duplicates(subset=[dset['id_col']])

        # Compute sex statistics
        sex_counts = filtered_df[dset['sex_col']].astype(str).value_counts()

        male_key = str(dset['sex_mapping']['Male'])
        female_key = str(dset['sex_mapping']['Female'])

        n_males = sex_counts.get(male_key, 0)
        n_females = sex_counts.get(female_key, 0)

        # Count scans vs subjects
        n_scans = len(filtered_df)
        n_subj = filtered_df[dset['id_col']].nunique()
 
        # Save training info if needed
        if dset['set'] in ['training', 'pretraining']:
            for _, row in filtered_df.iterrows():
                training_data.append({
                    'id': row[dset['id_col']],
                    'age': row[dset['age_col']]
                })
            training_sex_counts['Female'] += n_females
            training_sex_counts['Male'] += n_males
        
        # Add row
        new_row = {
            'repository' : name,
            'set' : dset['set'],
            'N_subj' : n_subj,
            'N_scans' : n_scans,
            'min' : f'{filtered_df[dset["age_col"]].min():.1f}',
            'max' : f'{filtered_df[dset["age_col"]].max():.1f}',
            'μ' : f'{filtered_df[dset["age_col"]].mean():.1f}',
            'σ' : f'{filtered_df[dset["age_col"]].std():.1f}',
            'M:F' : f'1 / {n_females/n_males:.1f}' if n_males > 0 else 'NA'
        }

        d_table = pd.concat([d_table, pd.DataFrame([new_row])], ignore_index=True)

    # Add combined training stats row
    if training_data:
        train_df = pd.DataFrame(training_data)

        if dset['set'] == 'pretraining': 
            set_name = 'All Pretraining'
        else: 
            set_name = 'All Training'

        n_scans = len(train_df)
        n_subj = train_df['id'].nunique()

        combined_row = {
            'repository' : set_name,
            'set' : 'combined',
            'N_subj' : n_subj,
            'N_scans' : n_scans,
            'min' : f'{train_df["age"].min():.1f}',
            'max' : f'{train_df["age"].max():.1f}',
            'μ' : f'{train_df["age"].mean():.1f}',
            'σ' : f'{train_df["age"].std():.1f}',
            'M:F' : f'1 / {training_sex_counts["Female"]/training_sex_counts["Male"]:.1f}' if training_sex_counts["Male"] > 0 else 'NA'
        }

        d_table = pd.concat([d_table, pd.DataFrame([combined_row])], ignore_index=True)
    
    return d_table

# Get the dataset statistics
stats = get_dataset_statistics(cfg)
stats.style.hide(axis='index')

repository,set,N_subj,N_scans,min,max,μ,σ,M:F
UKBB,training,9619,9619,45.5,82.4,64.8,7.8,1 / 1.1
IXI,training,480,480,20.0,86.3,50.9,16.1,1 / 1.3
NACC,training,3176,4324,18.9,100.2,69.2,10.9,1 / 2.0
ADNI_CN,testing,517,1129,55.5,104.3,75.7,6.8,1 / 1.0
ADNI_MCI,testing,83,153,56.5,95.9,73.7,7.7,1 / 1.2
ADNI_AD,testing,354,477,55.2,93.0,76.1,8.1,1 / 0.9
All Training,combined,13275,14423,18.9,100.2,65.7,9.8,1 / 1.3


Train the model using the optimal params

In [16]:
'''
To find optimal params, use find_model_params.py

Update config based on this

Then, run train_gnn.py
'''

'\nTo find optimal params, use find_model_params.py\n\nUpdate config based on this\n\nThen, run train_gnn.py\n'

Get the model predictions for CNs with no processing for supplementary figure

In [13]:
import torch
sys.path.append('/mnt/md0/tempFolder/samAnderson/unet-gnn/functions/')
from nn_optim_unet import gnn_builder, test_nn
from postprocessing import PostProcessor

# Load model
model = gnn_builder(feature_sizes=cfg.feature_sizes, dropout_levels=cfg.dropout_levels)
model.load_state_dict(torch.load(cfg.model_weights))

# Generate predictions
test_dict = test_nn(model, cfg.unproc_config)

# Visualize LBAs without any processing
lbags = test_dict['predictions'] - test_dict['targets'][:, None]
    
# Create LBAG plots (avg across subjects)
p = PostProcessor(first=cfg.first)
p.generate_cortical_plot(lbags.mean(axis=0), save_to=f'{cfg.unproc_config.vis_path}unprocessed_lbags', medial_present=True);

# (Later stages for figure are made in 2_pathology_analysis.ipynb)

MAE (L1) Loss: 6.354 across 1128 observations
